In [1]:
# ============================================================
# Notebook    : 11_governance_fairness_case_b.ipynb
# Description : Fairness audit for Case B (B2 LightGBM-Full),
#               the only Case with genuine protected attributes
#               (GENDER, RACE) rather than a proxy cohort.
#
#               Three metrics computed, each answering a
#               different question:
#                 - Demographic Parity : do groups get flagged
#                   positive at similar RATES? (doesn't account
#                   for true risk differing by group)
#                 - Equalized Odds     : do groups have similar
#                   TPR/FPR? (the main metric — separates true
#                   risk differences from model bias)
#                 - Calibration        : does the same predicted
#                   probability mean the same thing across groups?
#
#               Uses the EXACT test set saved by notebook 09
#               (case_b_X_test_for_fairness.csv/y_test...csv) so
#               rows align 1:1 with the SHAP analysis already done.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm scikit-learn pandas numpy matplotlib


# ============================================================
# 1. Common imports
# ============================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

np.random.seed(42)


# ============================================================
# 2. Load the EXACT test set used in notebook 09's SHAP analysis
#    (not re-split — guarantees row-for-row alignment)
# ============================================================
X_test = pd.read_csv("data/sequences/case_b_X_test_for_fairness.csv")
y_test = pd.read_csv("data/sequences/case_b_y_test_for_fairness.csv").iloc[:, 0]

print(f"[CHECK 1] X_test shape: {X_test.shape}")
print(f"[CHECK 1] y_test shape: {y_test.shape}")
print(f"[CHECK 1] Positive rate: {y_test.mean()*100:.2f}%")

CAT_COLS_B = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
    "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE",
]
for col in CAT_COLS_B:
    X_test[col] = X_test[col].astype("category")


# ============================================================
# 3. Load model (same as notebook 09) and get predictions
# ============================================================
model = lgb.Booster(model_file="data/sequences/case_b_lightgbm_full_model.txt")
y_pred_prob = model.predict(X_test)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

print(f"\n[CHECK 2] Overall AUC context (for reference, matches 06b): "
      f"predicted positive rate = {y_pred_cls.mean()*100:.2f}%")


# ============================================================
# 4. Cohort setup — GENDER and RACE separately, plus a combined
#    view. RACE has a known 9:1 imbalance (9,012 majority vs 988
#    minority from notebook 05's EDA) — flagged explicitly since
#    small subgroup sizes make metric estimates noisier.
# ============================================================
print(f"\n[CHECK 3] GENDER distribution in test set:")
print(X_test["GENDER"].value_counts())
print(f"\n[CHECK 3] RACE distribution in test set:")
print(X_test["RACE"].value_counts())
print(f"\n[CHECK 3] NOTE: RACE minority subgroup is small — treat its")
print(f"          metric estimates with wider uncertainty than GENDER's.")


# ============================================================
# 5. Metric functions
# ============================================================
def compute_group_metrics(y_true, y_pred_cls, y_pred_prob, group_labels):
    """
    For each group, compute:
      - n, actual positive rate (base rate)
      - Demographic Parity component: predicted positive rate
      - Equalized Odds components: TPR, FPR
      - Calibration component: mean predicted prob among
        predicted-positive cases vs actual positive rate there
    """
    results = []
    for g in sorted(pd.Series(group_labels).unique()):
        mask = (group_labels == g)
        yt = y_true[mask]
        yp = y_pred_cls[mask]
        ypp = y_pred_prob[mask]

        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        results.append({
            "group": g,
            "n": mask.sum(),
            "actual_positive_rate": yt.mean(),
            "predicted_positive_rate": yp.mean(),   # Demographic Parity metric
            "TPR": tpr,                              # Equalized Odds metric 1
            "FPR": fpr,                              # Equalized Odds metric 2
            "mean_pred_prob": ypp.mean(),
        })
    return pd.DataFrame(results)


def compute_calibration(y_true, y_pred_prob, group_labels, n_bins=5):
    """
    Bin predictions into n_bins by predicted probability, then
    for each (group, bin) compare mean predicted prob to actual
    positive rate. Large gaps indicate the same score means
    different things across groups.
    """
    bins = pd.qcut(y_pred_prob, n_bins, duplicates="drop")
    df = pd.DataFrame({
        "y_true": y_true.values if hasattr(y_true, "values") else y_true,
        "y_pred_prob": y_pred_prob,
        "group": group_labels,
        "bin": bins,
    })
    calib = df.groupby(["group", "bin"], observed=True).agg(
        n=("y_true", "size"),
        mean_pred_prob=("y_pred_prob", "mean"),
        actual_positive_rate=("y_true", "mean"),
    ).reset_index()
    calib["calibration_gap"] = calib["mean_pred_prob"] - calib["actual_positive_rate"]
    return calib


# ============================================================
# 6. GENDER audit
# ============================================================
print("\n" + "="*60)
print("GENDER — Fairness Audit")
print("="*60)

gender_metrics = compute_group_metrics(
    y_test.values, y_pred_cls, y_pred_prob, X_test["GENDER"].values
)
print("\n[GENDER] Group metrics:")
print(gender_metrics.to_string(index=False))

dp_gap_gender = gender_metrics["predicted_positive_rate"].max() - gender_metrics["predicted_positive_rate"].min()
tpr_gap_gender = gender_metrics["TPR"].max() - gender_metrics["TPR"].min()
fpr_gap_gender = gender_metrics["FPR"].max() - gender_metrics["FPR"].min()

print(f"\n[GENDER] Demographic Parity gap (predicted positive rate): {dp_gap_gender:.4f}")
print(f"[GENDER] Equalized Odds — TPR gap: {tpr_gap_gender:.4f}")
print(f"[GENDER] Equalized Odds — FPR gap: {fpr_gap_gender:.4f}")

calib_gender = compute_calibration(y_test, y_pred_prob, X_test["GENDER"].values)
print(f"\n[GENDER] Calibration by bin (gap = mean_pred_prob - actual_positive_rate):")
print(calib_gender.to_string(index=False))


# ============================================================
# 7. RACE audit
# ============================================================
print("\n" + "="*60)
print("RACE — Fairness Audit")
print("="*60)

race_metrics = compute_group_metrics(
    y_test.values, y_pred_cls, y_pred_prob, X_test["RACE"].values
)
print("\n[RACE] Group metrics:")
print(race_metrics.to_string(index=False))

dp_gap_race = race_metrics["predicted_positive_rate"].max() - race_metrics["predicted_positive_rate"].min()
tpr_gap_race = race_metrics["TPR"].max() - race_metrics["TPR"].min()
fpr_gap_race = race_metrics["FPR"].max() - race_metrics["FPR"].min()

print(f"\n[RACE] Demographic Parity gap (predicted positive rate): {dp_gap_race:.4f}")
print(f"[RACE] Equalized Odds — TPR gap: {tpr_gap_race:.4f}")
print(f"[RACE] Equalized Odds — FPR gap: {fpr_gap_race:.4f}")

calib_race = compute_calibration(y_test, y_pred_prob, X_test["RACE"].values)
print(f"\n[RACE] Calibration by bin (gap = mean_pred_prob - actual_positive_rate):")
print(calib_race.to_string(index=False))


# ============================================================
# 8. Visualization — TPR/FPR by group, both attributes
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (metrics_df, title) in zip(
    axes, [(gender_metrics, "GENDER"), (race_metrics, "RACE")]
):
    x = np.arange(len(metrics_df))
    width = 0.35
    ax.bar(x - width/2, metrics_df["TPR"], width, label="TPR")
    ax.bar(x + width/2, metrics_df["FPR"], width, label="FPR")
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_df["group"])
    ax.set_title(f"{title} — TPR/FPR by group")
    ax.set_ylabel("Rate")
    ax.legend()

plt.tight_layout()
plt.savefig("data/sequences/fairness_case_b_tpr_fpr.png")
plt.close()
print("\nSaved: data/sequences/fairness_case_b_tpr_fpr.png")


# ============================================================
# 9. Save raw metrics for the write-up
# ============================================================
gender_metrics.to_csv("data/sequences/fairness_case_b_gender_metrics.csv", index=False)
race_metrics.to_csv("data/sequences/fairness_case_b_race_metrics.csv", index=False)
calib_gender.to_csv("data/sequences/fairness_case_b_gender_calibration.csv", index=False)
calib_race.to_csv("data/sequences/fairness_case_b_race_calibration.csv", index=False)
print("Saved: fairness_case_b_{gender,race}_metrics.csv, "
      "fairness_case_b_{gender,race}_calibration.csv")


# ============================================================
# 10. Summary
# ============================================================
print("\n===== Case B Fairness Audit Summary =====")
print(f"GENDER — DP gap: {dp_gap_gender:.4f} | TPR gap: {tpr_gap_gender:.4f} | FPR gap: {fpr_gap_gender:.4f}")
print(f"RACE   — DP gap: {dp_gap_race:.4f} | TPR gap: {tpr_gap_race:.4f} | FPR gap: {fpr_gap_race:.4f}")
print(f"\nNOTE: RACE minority subgroup n={race_metrics[race_metrics['group']!='majority']['n'].values[0] if 'majority' in race_metrics['group'].values else 'see table'} "
      f"— interpret RACE gaps with wider uncertainty than GENDER's.")
print(f"Recall from SHAP (notebook 09): GENDER mean|SHAP|=0.230 (rank 4), "
      f"RACE mean|SHAP|=0.010 (rank 12) — SHAP magnitude gives a prior")
print(f"expectation that GENDER gaps are more likely to be substantive")
print(f"than RACE gaps, though this must be confirmed by the metrics above,")
print(f"not assumed from SHAP alone.")
print("===========================================")

[CHECK 1] X_test shape: (1500, 18)
[CHECK 1] y_test shape: (1500,)
[CHECK 1] Positive rate: 31.33%

[CHECK 2] Overall AUC context (for reference, matches 06b): predicted positive rate = 38.13%

[CHECK 3] GENDER distribution in test set:
GENDER
male      776
female    724
Name: count, dtype: int64

[CHECK 3] RACE distribution in test set:
RACE
majority    1338
minority     162
Name: count, dtype: int64

[CHECK 3] NOTE: RACE minority subgroup is small — treat its
          metric estimates with wider uncertainty than GENDER's.

GENDER — Fairness Audit

[GENDER] Group metrics:
 group   n  actual_positive_rate  predicted_positive_rate      TPR      FPR  mean_pred_prob
female 724              0.266575                 0.366022 0.854922 0.188324        0.362199
  male 776              0.356959                 0.395619 0.805054 0.168337        0.425462

[GENDER] Demographic Parity gap (predicted positive rate): 0.0296
[GENDER] Equalized Odds — TPR gap: 0.0499
[GENDER] Equalized Odds — FPR gap: